In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import wandb
import matplotlib.pyplot as plt
import os
import joblib

# Definimos dónde está el archivo y las reglas básicas de cómo va a estudiar el modelo
DATA_PATH = "global_house_purchase_dataset.csv" 
EPOCHS = 20
BATCH_SIZE = 1024
LEARNING_RATE = 0.001

def main():
    # Nos conectamos a Wand) para guardar el experimento
    wandb.login() 
    wandb.init(project="Predicción de compra de casas - Erick Sandoval Flores", config={
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE
    })
    
    # Cargamos la tabla de datos original
    df = pd.read_csv(DATA_PATH)
    
    # Quitamos el ID de la propiedad porque es un número de serie que no ayuda a predecir nada
    if 'property_id' in df.columns:
        df = df.drop(columns=['property_id'])
        
    # Separamos lo que queremos adivinar de las pistas que tenemos
    X = df.drop(columns=['decision'])
    y = df['decision'].values
    
    # Identificamos cuáles columnas son de texto y cuáles son de números
    categorical_cols = ['country', 'city', 'property_type', 'furnishing_status']
    numerical_cols = [col for col in X.columns if col not in categorical_cols]
    
    # Preparamos una herramienta que va a traducir los textos a números y q ajusta la escala de los números grandes para que la red no se confunda
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
        ]
    )
    
    y = y.astype(np.float32).reshape(-1, 1)
    
    # 70% para estudiar, 20% para hacer pruebas y 10% para el examen final aislado
    X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
    X_test_raw, X_val_raw, y_test, y_val = train_test_split(X_temp_raw, y_temp, test_size=(1/3), random_state=42, stratify=y_temp)
    
    # solo aprende las reglas viendo los datos de Train, Luego, usa esas mismas reglas para traducir los datos de prueba y validación.
    X_train_processed = preprocessor.fit_transform(X_train_raw)
    X_test_processed = preprocessor.transform(X_test_raw)
    X_val_processed = preprocessor.transform(X_val_raw)
    
    # Aseguramos que todos los datos tengan el formato matemático correcto
    if hasattr(X_train_processed, "toarray"):
        X_train_processed = X_train_processed.toarray()
        X_test_processed = X_test_processed.toarray()
        X_val_processed = X_val_processed.toarray()
        
    X_train = X_train_processed.astype(np.float32)
    X_test = X_test_processed.astype(np.float32)
    X_val = X_val_processed.astype(np.float32)
    
    print(f"Tamaños de datasets: Train: {X_train.shape[0]}, Test: {X_test.shape[0]}, Val: {X_val.shape[0]}")
    
    # Empaquetamos los datos para que PyTorch los pueda procesar por lotes
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    val_dataset = TensorDataset(torch.tensor(X_val), torch.tensor(y_val))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    input_dim = X_train.shape[1]
    
    # Aquí armamos la arquitectura de la red neuronal capa por capa, usamos dropout para obligar al modelo a mejorar y no solo memorizar.
    class HousePredictionNet(nn.Module):
        def __init__(self, input_dim):
            super(HousePredictionNet, self).__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )
            
        def forward(self, x):
            return self.net(x)
            
    model = HousePredictionNet(input_dim)
    
    
    # Guardamos un archivo de texto con el esquema de la red
    with open('network_architecture.txt', 'w') as f:
        f.write(str(model))
        
    # Que wandb haga su trabajo
    wandb.watch(model, log="all")
    
    # Configuramos la forma en la que el modelo va a medir sus errores y cómo va a corregirlos
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    # Listas para ir guardando el historial de calificaciones
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    
    print("Iniciando entrenamiento")
    for epoch in range(EPOCHS):
        
        # Fase de estudio
        model.train()
        epoch_train_loss = 0
        epoch_train_correct = 0
        total_train = 0
        
        # Pasamos los datos por bloques
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)         # El modelo intenta adivinar
            loss = criterion(outputs, batch_y) # Vemos qué tanto se equivocó
            loss.backward()                  # Entiende de dónde vino el error
            optimizer.step()                 # Ajusta sus conexiones matemáticas
            
            # Vamos sumando las calificaciones del estudio
            epoch_train_loss += loss.item() * batch_X.size(0)
            preds = torch.round(torch.sigmoid(outputs))
            epoch_train_correct += (preds == batch_y).sum().item()
            total_train += batch_X.size(0)
            
        train_loss = epoch_train_loss / total_train
        train_acc = epoch_train_correct / total_train
        
        # Fase de prueba
        # Ponemos el modelo en modo de evaluación para que no aprenda de estos datos, solo demuestre lo que sabe
        model.eval()
        epoch_test_loss = 0
        epoch_test_correct = 0
        total_test = 0
        
        with torch.no_grad(): # Apagamos el cálculo de errores para ahorrar memoria
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Sumamos las calificaciones de la prueba
                epoch_test_loss += loss.item() * batch_X.size(0)
                preds = torch.round(torch.sigmoid(outputs))
                epoch_test_correct += (preds == batch_y).sum().item()
                total_test += batch_X.size(0)
                
        test_loss = epoch_test_loss / total_test
        test_acc = epoch_test_correct / total_test
        
        # Guardamos el historial localmente
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accs.append(train_acc)
        test_accs.append(test_acc)
        
        # Mandamos las calificaciones de esta vuelta a wandb
        wandb.log({
            "epoch": epoch+1,
            "train_loss": train_loss,
            "test_loss": test_loss,
            "train_accuracy": train_acc,
            "test_accuracy": test_acc
        })
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

    # Validación
    # Ya que terminó, lo probamos con el 10% de datos que estaba totalmente escondido
    model.eval()
    val_correct = 0
    total_val = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            outputs = model(batch_X)
            preds = torch.round(torch.sigmoid(outputs))
            val_correct += (preds == batch_y).sum().item()
            total_val += batch_X.size(0)
            
    val_accuracy = val_correct / total_val
    print(f"\n[!] PRECISION EN VALIDACION (VALIDATION ACCURACY): {val_accuracy:.4f} ({(val_accuracy*100):.2f}%)\n")
    wandb.log({"val_accuracy": val_accuracy})
    
    # Gráficas
    epochs_range = range(1, EPOCHS + 1)
    
    # Gráfica de costo
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_losses, label='Entrenamiento (Train)')
    plt.plot(epochs_range, test_losses, label='Prueba (Test)')
    plt.title('Función de Costo (Loss) vs Épocas')
    plt.xlabel('Épocas')
    plt.ylabel('Costo')
    plt.legend()
    plt.grid(True)
    plt.savefig('loss_graph.png')
    plt.show() 
    plt.close()
    
    # Gráfica de precisión
    plt.figure(figsize=(10, 6))
    plt.plot(epochs_range, train_accs, label='Entrenamiento (Train)')
    plt.plot(epochs_range, test_accs, label='Prueba (Test)')
    plt.title('Precisión (Accuracy) vs Épocas')
    plt.xlabel('Épocas')
    plt.ylabel('Precisión (Accuracy)')
    plt.legend()
    plt.grid(True)
    plt.savefig('accuracy_graph.png')
    plt.show() 
    plt.close()
    
    # Guardamos los pesos del modelo 
    print("Guardando el modelo y preprocesador...")
    torch.save(model.state_dict(), 'predicccionCompraCasas.pth')
    joblib.dump(preprocessor, 'preprocessor.joblib')
    
    wandb.finish() 
    
if __name__ == '__main__':
    main()